# Exercice 2 — Calibrer un taux de rachat par descente de gradient

##  Contexte métier
Vous calibrez un **modèle de rachat dynamique** : le taux de rachat annuel (en %) dépend de l'**écart de taux** = taux de marché − taux servi par l'assureur (en points de %). Plutôt que d'utiliser une formule toute faite, vous allez **entraîner le modèle à la main** par descente de gradient — exactement ce que font les librairies de ML en coulisses.

##  Objectifs
- Écrire une fonction de coût (erreur quadratique moyenne) d'un modèle linéaire de rachat
- Calculer son gradient et le **valider numériquement**
- Implémenter la descente de gradient et la comparer à la solution exacte des moindres carrés
- Observer les trois régimes du taux d'apprentissage η (trop petit / adapté / trop grand)
- Constater qu'un minimum **local** peut piéger la descente (fonction non convexe)

##  Rappels théoriques mobilisés (slides de la session)
- *Fonction de coût et gradient : le moteur de l'apprentissage* — pas contre la pente, minimum local
- *Le taux d'apprentissage η : le bouton de réglage critique* — trois régimes
- *Métriques : ce que l'on mesure décide de ce que l'on obtient* — MSE = pénalisation des grosses erreurs
- *Formulaire minimal* — `θ ← θ − η ∇J(θ)` et MSE

##  Mini-dictionnaire & prérequis
| Élément | Signification / usage |
|---|---|
| `X` | matrice (n×2) : colonne de 1 (intercept) + écart de taux |
| `y` | taux de rachat observé (%) pour chacune des 60 observations |
| `θ = [a, b]` | paramètres : taux de base `a` et sensibilité `b` à l'écart de taux |
| `J(θ)` = MSE | J(θ) = (1/n) Σ (Xθ − y)² |
| Gradient | ∇J(θ) = (2/n) · Xᵀ(Xθ − y) |
| η (`lr`) | taux d'apprentissage : taille du pas de descente |
| `np.linalg.lstsq` | solution exacte des moindres carrés (référence) |

## Mode d'emploi
- Complétez les cellules marquées `...  # À COMPLÉTER` puis exécutez-les **dans l'ordre**.
- Les cellules **Vérification** vous disent immédiatement si vous êtes sur la bonne piste (✅ / ❌ + indice).
- Bloqué ? Ouvrez les **💡 indices** progressifs avant de demander la correction.
- À la fin, la cellule `bilan()` donne votre score.

In [ ]:
# ✅ Outil d'auto-vérification — NE PAS MODIFIER
import numpy as np, pandas as pd, matplotlib.pyplot as plt, hashlib
_resultats = {}
def verifier(nom, condition, indice=""):
    """Évalue `condition` (une fonction sans argument) ; affiche ✅ ou ❌ + indice."""
    try: ok = bool(condition())
    except Exception: ok = False
    _resultats[nom] = ok
    print(("✅ " if ok else "❌ ") + nom + ("" if ok or not indice else "\n    💡 " + indice))
def bilan():
    n, k = len(_resultats), sum(_resultats.values())
    print(f"\n🏁 Score : {k}/{n} vérifications réussies")
    print("   Bravo, exercice terminé !" if k == n and n > 0 else "   Reprenez les ❌ ci-dessus (relancez la cellule concernée puis `bilan()`).")
def _h(s): return hashlib.sha1(s.encode()).hexdigest()[:8]
%matplotlib inline

---
## Partie A — Les données et le modèle

In [ ]:
rng = np.random.default_rng(2024)
n = 60
ecart = rng.uniform(-0.5, 3.0, n)                          # écart de taux (points de %)
y = 3.5 + 2.0 * ecart + rng.normal(0, 0.7, n)              # taux de rachat observé (%)
y = np.clip(y, 0.2, None)
X = np.column_stack([np.ones(n), ecart])                   # colonne de 1 + écart
theta_mco = np.linalg.lstsq(X, y, rcond=None)[0]           # solution exacte (référence)
plt.scatter(ecart, y, alpha=.7); plt.xlabel("écart de taux (pts)"); plt.ylabel("taux de rachat (%)")
plt.title("Rachats observés vs écart de taux"); plt.show()
print("Solution exacte (référence) :", theta_mco.round(3))

**Q1 · Fonction de coût.** Écrivez `cout(theta, X, y)` qui renvoie la MSE : moyenne des carrés de `X @ theta − y`.

<details><summary>💡 Indice 1</summary>

Prédictions : `X @ theta`. Erreurs : `X @ theta - y`.

</details>

<details><summary>💡 Indice 2</summary>

`np.mean((X @ theta - y) ** 2)`

</details>

In [ ]:
def cout(theta, X, y):
    ...   # À COMPLÉTER
print("J(0,0) =", cout(np.zeros(2), X, y))

In [ ]:
verifier("Q1 · J(0,0) correct", lambda: np.isclose(cout(np.zeros(2), X, y), 35.006129, rtol=1e-6))
verifier("Q1 · J minimal à la solution exacte", lambda: cout(theta_mco, X, y) < cout(theta_mco + np.array([0.3, -0.2]), X, y))


## Partie B — Le gradient

**Q2 · Gradient.** Écrivez `gradient(theta, X, y)` = `(2/n) · Xᵀ (Xθ − y)`. La cellule de vérification le compare à une **dérivée numérique** (différences finies) : c'est le réflexe de test d'un data scientist.

<details><summary>💡 Indice 1</summary>

`X.T @ (X @ theta - y)` donne un vecteur de taille 2 ; multipliez par `2 / len(y)`.

</details>

In [ ]:
def gradient(theta, X, y):
    ...   # À COMPLÉTER

In [ ]:
def _grad_num(theta, h=1e-6):
    g = np.zeros_like(theta)
    for i in range(len(theta)):
        e = np.zeros_like(theta); e[i] = h
        g[i] = (cout(theta + e, X, y) - cout(theta - e, X, y)) / (2 * h)
    return g
t_test = np.array([1.0, 0.5])
verifier("Q2 · gradient de dimension 2", lambda: np.shape(gradient(t_test, X, y)) == (2,))
verifier("Q2 · gradient = dérivée numérique", lambda: np.allclose(gradient(t_test, X, y), _grad_num(t_test), rtol=1e-4), "vérifiez le facteur 2/n et la transposée de X")
verifier("Q2 · gradient nul à la solution exacte", lambda: np.allclose(gradient(theta_mco, X, y), 0, atol=1e-8))

## Partie C — La descente de gradient

**Q3 · Descente.** Complétez `descente(X, y, lr, n_iter)` : on part de `θ = (0, 0)`, et à chaque itération `θ ← θ − lr · ∇J(θ)`. La fonction renvoie `theta` final et la liste `historique` des coûts (un par itération, **avant** la mise à jour).

<details><summary>💡 Indice 1</summary>

Dans la boucle : `historique.append(cout(theta, X, y))`, puis `theta = theta - lr * gradient(theta, X, y)`.

</details>

In [ ]:
def descente(X, y, lr, n_iter):
    theta = np.zeros(X.shape[1])
    historique = []
    for _ in range(n_iter):
        ...   # À COMPLÉTER : enregistrer le coût puis faire un pas
    return theta, historique

theta_gd, hist = descente(X, y, lr=0.1, n_iter=500)
print("θ (descente)  :", theta_gd.round(3)); print("θ (exact)     :", theta_mco.round(3))

In [ ]:
verifier("Q3 · historique de 500 coûts", lambda: len(hist) == 500)
verifier("Q3 · le coût décroît", lambda: hist[-1] < hist[0] and np.all(np.diff(hist) <= 1e-12))
verifier("Q3 · θ proche de la solution exacte", lambda: np.allclose(theta_gd, theta_mco, atol=0.02), "augmentez n_iter ou vérifiez le signe du pas")

## Partie D — Le taux d'apprentissage η

**Q4.** Lancez la descente avec trois valeurs de η : `0.001`, `0.1`, `0.4` (200 itérations). Complétez la fonction `diagnostic(hist, cout_opt)` qui renvoie :
- `"trop grand"` si le coût final est **supérieur** au coût initial (divergence),
- `"adapté"` si le coût final est à moins de 5 % du coût optimal `cout_opt`,
- `"trop petit"` sinon.

<details><summary>💡 Indice 1</summary>

Ordre des tests : d'abord la divergence (`hist[-1] > hist[0]`), puis la proximité à l'optimum (`hist[-1] <= 1.05 * cout_opt`).

</details>

In [ ]:
cout_opt = cout(theta_mco, X, y)
def diagnostic(hist, cout_opt):
    ...   # À COMPLÉTER

regimes = {}
plt.figure(figsize=(7, 3.5))
for lr in [0.001, 0.1, 0.4]:
    _, h = descente(X, y, lr, 200)
    regimes[lr] = diagnostic(h, cout_opt)
    plt.plot(h, label=f"η={lr} → {regimes[lr]}")
plt.yscale("log"); plt.xlabel("itération"); plt.ylabel("J(θ) (échelle log)"); plt.legend(); plt.show()
print(regimes)

In [ ]:
verifier("Q4 · η = 0,001 est trop petit", lambda: regimes.get(0.001) == "trop petit", "après 200 pas le coût est encore loin de l'optimum")
verifier("Q4 · η = 0,1 est adapté", lambda: regimes.get(0.1) == "adapté")
verifier("Q4 · η = 0,4 diverge", lambda: regimes.get(0.4) == "trop grand", "le coût final dépasse le coût initial")

## Partie E — Minimum local vs global (bonus)

Reprenons la fonction de la slide : `J(θ) = θ² + 0,8·sin(3θ)`, dont la dérivée est `J'(θ) = 2θ + 2,4·cos(3θ)`.

**Q5.** Écrivez `dJ(theta)` puis `descente_1d(theta0, lr=0.05, n_iter=300)` qui renvoie le θ final. Lancez-la depuis `θ₀ = 2` (→ `theta_depart_2`) et depuis `θ₀ = −1,5` (→ `theta_depart_m15`).

In [ ]:
J = lambda t: t**2 + 0.8 * np.sin(3 * t)
def dJ(theta):
    ...   # À COMPLÉTER

def descente_1d(theta0, lr=0.05, n_iter=300):
    ...   # À COMPLÉTER : renvoie le theta final

theta_depart_2 = ...      # À COMPLÉTER
theta_depart_m15 = ...    # À COMPLÉTER
tt = np.linspace(-2.5, 2.5, 400); plt.plot(tt, J(tt))
for t in (theta_depart_2, theta_depart_m15): plt.scatter([t], [J(t)], zorder=3)
plt.title("Deux départs, deux minima"); plt.show()
print(theta_depart_2, theta_depart_m15)

In [ ]:
verifier("Q5 · dérivée correcte", lambda: np.isclose(dJ(0.7), 2*0.7 + 2.4*np.cos(2.1)))
verifier("Q5 · départ à 2 → minimum local", lambda: np.isclose(theta_depart_2, 1.146813, atol=1e-2))
verifier("Q5 · départ à −1,5 → minimum global", lambda: np.isclose(theta_depart_m15, -0.407969, atol=1e-2))
verifier("Q5 · le minimum atteint depuis 2 est moins bon", lambda: J(theta_depart_2) > J(theta_depart_m15) + 0.5)

In [ ]:
bilan()